# **4.x Notebook**

Comprehensive (hopefully) notebook for project up to the current point. WIP
by Jack Phelan  

## Table of Contents
- [Setup](#1-setup)
- [Data Loading and mlflow Integration](#2-data-loading-and-mlflow-integration)


## **1. Setup**

In [29]:
# ── standard library ─────────────────────────────────────────────────────────
import re
import sys
import json
import datetime
from contextlib import contextmanager
from pathlib import Path
from types import SimpleNamespace

# ── data & numeric ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ── ML ────────────────────────────────────────────────────────────────────────
import gcsfs
import joblib
import mord
import lightgbm as lgb
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    roc_auc_score,
    f1_score,
    roc_curve,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import mlflow
import mlflow.data
from factor_analyzer import FactorAnalyzer



# ── project ───────────────────────────────────────────────────────────────────
from src.modeling.eval import evaluate_binary as evaluate, evaluate_multiclass as evaluate_mc, tag_comparison_df as _tag
from src.features import raw_data_loader, feature_encoder
from src.features.transformers import Factor_Analyzer_Transformer
from src.modeling.fa_tuning import tune_factor_analysis_rotation, build_best_factor_analysis_pipeline
from src.utils.data_utils import convert_ahi, make_multiclass_y


## **2. Data Loading and mlflow Integration**




### 2.1.0 Loading default df and applying EDA preprocessing steps

In [30]:
# This is the default 1.0 dataset that will be iterated upon in future experiments.
raw_df = raw_data_loader.load_and_clean_raw("../..")
encoded_df = feature_encoder.encode_features(raw_df)
encoded_df = encoded_df.drop(columns=["ess_total_score", "isi_total_score"])
encoded_df.to_csv("../../data/processed/mlflow/mlflow_dataset_v1.csv", index=False)

### 2.1.1 Creating factor transformed dataset for 1.0 dataset

In [31]:
# performing factor analysis
factor_analyzer = FactorAnalyzer(n_factors=18, rotation="quartimax")

encoded_df  = pd.read_csv("../../data/processed/mlflow/mlflow_dataset_v1.csv")
encoded_features = encoded_df.drop(columns=["ahi"])
encoded_target = encoded_df["ahi"]
factor_analyzer.fit(encoded_features)
fa_loadings = pd.DataFrame(factor_analyzer.loadings_, index=encoded_features.columns
)

In [32]:
# display top n features per factor loading
n = 7

top_features = {}
for factor in fa_loadings.columns:
    top = fa_loadings[factor].abs().nlargest(n)
    top_features[factor] = [f"{feat}  ({fa_loadings.loc[feat, factor]:+.2f})" for feat in top.index]

pd.DataFrame(top_features, index=[f"#{i+1}" for i in range(n)])


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
#1,fati_interferes_with_work_fam_social_fss (+0.90),diff_staying_asleep_isi (+0.75),not_able_to_control_worrying_freq_gad (+0.75),ess_chance_of_dozing_inactive_public (+0.62),map_index_1_apnea_score (+1.01),feel_sleep_not_sound_isq (+0.78),troub_brth_nose (+0.84),age (+0.59),family_history_of_psychiatric_treatment_mdhx ...,map_index_4_narcolepsy_like_symptoms_score (+...,bmi (+0.77),sleep_affected_your_social_life_isq (+0.47),dream_recall_frequency_rarely_or_never (-0.74),sex (+0.40),ess_chance_of_dozing_lying_down_in_afternoon_w...,latency_sw (+0.42),never_feel_rested (-0.33),difficulty_watching_a_movie_fosq (+0.41)
#2,fati_causes_freq_probs_for_me_fss (+0.89),troub_slp_nose (+0.67),worry_too_much_diff_things_freq_gad (+0.72),ess_chance_of_dozing_in_a_car_while_stopped_fo...,snorting_gasping_map (+0.81),difficulty_staying_asleep_isq (+0.76),nasal_blockage_obstr_nose (+0.80),hypercholesterolemia_mdhx (+0.48),family_history_of_depression_mdhx (+0.64),hh_frequency_score (+0.73),map_score (+0.68),sleep_affected_other_important_parts_of_life_i...,dream_recall_frequency_infrequent (+0.68),morning_headache_map (+0.26),frequent_naps (+0.42),diff_falling_asleep_isi (+0.34),rmeq_total_score (+0.31),ess_chance_of_dozing_tv (-0.35)
#3,fati_interferes_with_responsibilities_fss (+0...,diff_falling_asleep_isi (+0.64),feeling_anxious_freq_gad (+0.67),ess_chance_of_dozing_sitting_quietly_after_lun...,breathing_stops_map (+0.72),multiple_awakenings_isq (+0.74),nasal_congestion_stuffiness_nose (+0.74),hypertension_mdhx (+0.45),family_history_of_anxiety_mdhx (+0.58),sleep_paralysis_frequency_score (+0.67),map_likelihood_ration (+0.52),sleep_affected_work_isq (+0.44),never_smoked (+0.23),feel_depressed_freq_phq (-0.26),difficulty_operating_motor_vehicles_for_short_...,diff_falling_asleep_isq (+0.33),feel_sleep_unrefreshing_isq (-0.29),difficulty_being_active_in_evening_fosq (+0.32)
#4,fati_top_three_disabling_sympts_fss (+0.85),diff_falling_asleep_isq (+0.62),feel_as_if_awful_might_happen_freq_gad (+0.60),fall_asleep_driving_map (+0.57),loud_snoring_map (+0.67),feel_sleep_unrefreshing_isq (+0.74),not_enough_air_excercise_nose (+0.62),cardiovascular_problem_other_mdhx (+0.41),family_history_of_other_psychiatric_illness_md...,fall_asleep_driving_map (+0.28),hypertension_mdhx (+0.31),sleep_made_you_irritable_isq (+0.40),frequent_naps (+0.17),thought_you_would_be_better_dead_freq_phq (-0...,ess_chance_of_dozing_sitting_and_reading (+0.29),how_satisfied_with_curr_sleep_pattern_isi (-0...,difficulty_staying_asleep_isq (+0.22),rmeq_total_score (-0.29)
#5,fati_interferes_with_phys_function_fss (+0.84),how_satisfied_with_curr_sleep_pattern_isi (+0...,feel_bad_abt_yourself_freq_phq (+0.58),map_index_3_excessive_daytime_sleepiness_ (+0...,map_score (+0.60),sleep_made_you_fatigued_isq (+0.47),allergies_or_sinus_problems_mdhx (+0.42),medical_problem_or_surgery_other_mdhx (+0.41),family_history_of_insomnia_mdhx (+0.32),map_index_3_excessive_daytime_sleepiness_ (+0...,exercise_amt_or_time (-0.27),sleep_caused_trouble_concentrating_isq (+0.39),frequent_daytime_sleepiness (+0.15),gastrointestinal_problem_or_surgery_mdhx (+0.23),difficulty_operating_motor_vehicles_for_long_d...,sex (+0.21),difficulty_being_active_in_morning_fosq (+0.22),effect_on_desire_for_intimacy_or_sex_fosq (+0...
#6,fati_prevents_sustained_phys_func_fss (+0.83),slp_quality_sw (-0.54),trouble_relaxing_freq_gad (+0.57),difficulty_operating_motor_vehicles_for_short_...,frequent_tossing_turning_map (+0.31),sleep_affected_your_social_life_isq (+0.30),ear_nose_and_throat_problem_or_surgery_other_m...,urologic_or_kidney_problem_mdhx (+0.34),family_history_of_fibromyalgia_or_chronic_fati...,morning_headache_map (+0.24),type_2_diabetes_mdhx (+0.22),sleep_made_you_fatigued_isq (+0.25),family_history_of_sleep_apnea_mdhx (+0.13),little_interest_or_pleasure_freq_phq (-0.21),ess_chance_of_dozing_tv (+0.25),rmeq_total_score (-0.20),feel_refreshed_after_nap (+0.22

In [33]:
fa_df = factor_analyzer.transform(encoded_features)
fa_df = pd.DataFrame(fa_df, columns=[f"Factor_{i+1}" for i in range(fa_df.shape[1])])
fa_df["ahi"] = encoded_target.values

fa_df.to_csv("../../data/processed/mlflow/mlflow_factor_dataset_v1.csv", index=False)

display(fa_df.head(5))


,Factor_1,Factor_2,Factor_3,Factor_4,Factor_5,Factor_6,Factor_7,Factor_8,Factor_9,Factor_10,Factor_11,Factor_12,Factor_13,Factor_14,Factor_15,Factor_16,Factor_17,Factor_18,ahi
0,35.766926,469.093860,-127.427710,911.196915,-3435.227408,-53.878825,199.330406,298.385842,-199.259447,651.468553,992.251477,34.591764,57.768046,77.941877,287.483926,-359.482086,142.346987,939.855590,0.3
1,323.854888,4195.291566,-1138.819720,8137.205140,-30744.337813,-490.166215,1790.764602,2676.898070,-1772.794579,5828.636113,8878.954965,310.146927,530.676634,686.210360,2569.776663,-3206.147546,1268.130313,8383.749697,0.4
2,-223.558856,-2914.739177,789.632700,-5658.463044,21369.494992,340.364690,-1246.176703,-1861.455930,1232.266396,-4053.484426,-6173.226013,-212.937160,-368.295586,-476.402247,-1787.902763,2229.786529,-881.470189,-5828.833304,1.0
3,-37.046131,-467.733177,128.692537,-909.541972,3433.387856,54.428891,-200.591382,-298.920519,198.294410,-649.286992,-992.362742,-34.220449,-59.547808,-75.840732,-287.886898,358.228540,-142.979521,-937.488593,3.9
4,41.470517,544.550645,-147.099404,1056.064349,-3993.485146,-64.505173,233.242283,347.471861,-229.042355,756.616701,1155.756794,40.741331,68.909986,89.147731,333.861565,-418.182822,165.603702,1086.883448,1.3


### 2.1.2 Creating CFA based dataset with v1 base

### 2.2 Logging Datasets in mlflow

In [34]:
# v1  dataset

encoded_dataset_path = "../../data/processed/mlflow/mlflow_dataset_v1.csv"
dataset = mlflow.data.from_pandas(
    encoded_df,
    source=encoded_dataset_path,
    name="encoded_dataset_v1",
    targets="ahi"
)

/opt/anaconda3/envs/py312_env/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


In [35]:
# fa v1 dataset

fa_dataset_path = "../../data/processed/mlflow/mlflow_factor_dataset_v1.csv"
dataset = mlflow.data.from_pandas(
    fa_df,
    source=fa_dataset_path,
    name="fa_dataset_v1",
    targets="ahi"
)

/opt/anaconda3/envs/py312_env/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
